# Importamos líbrerias

In [1]:
import pandas as pd

In [2]:
campaña = pd.read_csv('../data/raw/campaña.csv')
clientes = pd.read_csv('../data/raw/clientes.csv')
devoluciones = pd.read_csv('../data/raw/devoluciones.csv')
lineas_pedido = pd.read_csv('../data/raw/lineas_pedido.csv')
pedidos = pd.read_csv('../data/raw/pedidos.csv')
productos = pd.read_csv('../data/raw/productos.csv')
visitas_web = pd.read_csv('../data/raw/visitas_web.csv')

In [3]:
# tabla campañas
# Cambio de formato 
campaña['fecha_inicio'] = pd.to_datetime(campaña['fecha_inicio'], format='mixed', dayfirst=True, errors='coerce')
campaña['fecha_fin'] = pd.to_datetime(campaña['fecha_fin'], format='mixed', dayfirst=True, errors='coerce')

# Limpieza de valores "29,99 €"  ->  29.99
campaña['presupuesto'] = (
    campaña['presupuesto']
    .str.replace('€', '')
    .str.replace(',','.')
    .str.strip()
    .astype(float)
    )

campaña.dtypes

campana_id                      int64
nombre                            str
tipo                              str
fecha_inicio           datetime64[us]
fecha_fin              datetime64[us]
canal                             str
presupuesto                   float64
ventas_ano_anterior           float64
dtype: object

In [4]:
# Tabla clientes
# Formato datetime
clientes['fecha_registro'] = pd.to_datetime(clientes['fecha_registro'], format='mixed', dayfirst=True, errors='coerce')

# errores de texto
# Normalizar la categoría ciudad
antes_ciudad = clientes['ciudad'].nunique()
clientes['ciudad'] = clientes['ciudad'].str.strip().str.title()
despues_ciudad = clientes['ciudad'].nunique()

#Normalizar la categoria género a M y F
antes_genero = clientes['genero'].nunique()
GEN = {'f':'F', 'femenino':'F', 'm':'M','Maculino':'M'}
clientes['genero'] = clientes['genero'].str.upper().str.strip().replace({'MASCULINO':'M','FEMENINO':'F'})
despues_genero = clientes['genero'].nunique()

# Edades imposibles: menores a 16 o mayores a 90 (nadie compara con esas edades)
edad_malas = ((clientes['edad'] < 16)| (clientes['edad'] > 90)).sum()

# Conservar el último registo de cada cliente_id
clientes_limpio = clientes.drop_duplicates(subset='cliente_id', keep='last')

print('Tabla clientes: fecha_registo')
print(f'formato : {clientes['fecha_registro'].dtype}')
print('Tabla clientes: ciudad')
print(f'categorias crudas: {antes_ciudad}')
print(f'categorías normalizadas: {despues_ciudad}')
print('Tabla clientes: género')
print(f'categorias crudas: {antes_genero}')
print(f'categorías normalizadas: {despues_genero}')
print(f'Edades imposibles (<16 o >90): {edad_malas}')
print('Tabla clientes: cliente_id')
print(f'Clientes tras deduplicar: {len(clientes_limpio)}')


Tabla clientes: fecha_registo
formato : datetime64[us]
Tabla clientes: ciudad
categorias crudas: 31
categorías normalizadas: 11
Tabla clientes: género
categorias crudas: 6
categorías normalizadas: 2
Edades imposibles (<16 o >90): 18
Tabla clientes: cliente_id
Clientes tras deduplicar: 4970


In [5]:
#Tabla devoluciones
# formato Datetime
devoluciones['fecha_devolucion'] = pd.to_datetime(devoluciones['fecha_devolucion'], format='mixed', dayfirst=True, errors='coerce')

# errores de texto
# Normalizar la categoria talla
antes = devoluciones['talla'].nunique()
devoluciones['talla_norm'] = (devoluciones['talla']
        .astype(str).str.strip()
        .str.upper()
        .str.replace('TALLA','',regex=False)
)

print('Tabla devoluciones: fecha_devolución')
print(f'Formato: {devoluciones['fecha_devolucion'].dtype}')
print('Tabla devoluciones: talla')
print(f'Categoria talla antes (crudo): {antes}')
print(f'Categoria talla después: {devoluciones['talla_norm'].nunique()}')
print('Tabla devoluciones: Motivo')
print(f'Motivos distintos en crudo: {devoluciones['motivo'].nunique()}')



Tabla devoluciones: fecha_devolución
Formato: datetime64[us]
Tabla devoluciones: talla
Categoria talla antes (crudo): 29
Categoria talla después: 24
Tabla devoluciones: Motivo
Motivos distintos en crudo: 8


In [6]:
lineas_pedido['talla'].unique()

<StringArray>
[      '39',       'XL',        'L',        'S',        'M',  'Talla S',
        's',       '38',  'Talla M',        'm',       'XS',       '37',
 'Talla XS',        'l',       '36',       '41',  'Talla L',       '40',
 'Talla XL', 'Talla 36',       'xs',       '42',       'xl', 'Talla 39',
 'Talla 38', 'Talla 37', 'Talla 40', 'Talla 42', 'Talla 41']
Length: 29, dtype: str

In [7]:
# Tabla linea de pedido
# Cambio de formato 
lineas_pedido['precio_unitario'] =(
    lineas_pedido['precio_unitario']
    .str.replace('€','')
    .str.replace(',','.')
    .str.strip()
    .astype(float)
)

# Valores atípicos 
n_malas = (lineas_pedido['cantidad'] <= 0).sum()
# Cuenta las filas exactamente repetidas
n_dup = lineas_pedido.duplicated().sum()
lineas_pedido = lineas_pedido[lineas_pedido['cantidad'] > 0].drop_duplicates()

# Normalizar texto
# Columna descuento_pct
def normalizar(x):
    s =str(x).strip()
    if s.endswith('%'):         # "20%" -> 0.20
        return float(s[:-1])/100.00
    v = float(s)
    return v/100 if v > 1 else v # 20 -> 0.20 ; 0.2 -> 0.2

descuento_antes = lineas_pedido['descuento_pct'].nunique()
lineas_pedido['descuento_pct'] = lineas_pedido['descuento_pct'].apply(normalizar)
descuento_despues = lineas_pedido['descuento_pct'].nunique()

# Columna talla 
antes = lineas_pedido['talla'].nunique()
lineas_pedido['talla_norm'] = (lineas_pedido['talla']
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r'(?i)^talla\s*', '', regex=True) # Elimina 'Talla' o 'talla'
)

print('Tabla lineas de pedido: precio_unitario')
print(f'formato: {lineas_pedido['precio_unitario'].dtype}')
print('Tabla lineas de pedido: descuento_pct')
print(f'formato : {lineas_pedido['descuento_pct'].dtype}')
print('Tabla lineas de pedido: cantidad')
print(f"Cantidades invalidas (<=0): {n_malas}")
print(f"Filas duplicadas exactas: {n_dup}")
print(f"Lineas validas finales: {len(lineas_pedido)}")
print(f'Tabla lineas de pedido: talla')
print(f'Categoria talla antes (crudo): {antes}')
print(f'Categoria talla después: {lineas_pedido['talla_norm'].nunique()}')


Tabla lineas de pedido: precio_unitario
formato: float64
Tabla lineas de pedido: descuento_pct
formato : float64
Tabla lineas de pedido: cantidad
Cantidades invalidas (<=0): 15
Filas duplicadas exactas: 139
Lineas validas finales: 19986
Tabla lineas de pedido: talla
Categoria talla antes (crudo): 29
Categoria talla después: 12


In [20]:
# Tabla pedidos
# Cambio de formato 
pedidos['fecha_pedido'] = pd.to_datetime(pedidos['fecha_pedido'], format='mixed', dayfirst=True, errors='coerce')

# Normalizar texto 
# Columna ciudad
pedidos['ciudad'] = pedidos['ciudad'].str.strip().str.title()

print(f'Formato columna fecha_pedido: {pedidos['fecha_pedido'].dtype}')
print(f'ciudades canónicas {pedidos['ciudad'].unique()}')
print(f"Rango de fechas: {pedidos['fecha_pedido'].min().date()} a {pedidos['fecha_pedido'].max().date()}")

Formato columna fecha_pedido: datetime64[us]
ciudades canónicas <StringArray>
[   'Oporto',    'Lisboa',  'Zaragoza',    'Madrid',   'Sevilla',  'A Coruña',
 'Barcelona',  'Valencia',    'Bilbao',    'Málaga',    'Malaga']
Length: 11, dtype: str
Rango de fechas: 2024-01-07 a 2024-12-07


In [14]:
# Tabla producto
# Normalización de texto 
# Columna categoria
CANON = {
    'vestidos': 'Vestidos', 'vestido': 'Vestidos',
    'zapatos': 'Calzado', 'calzado': 'Calzado',
    'camiseta': 'Camisetas', 'camisetas': 'Camisetas',
    'pantalon': 'Pantalones', 'pantalones': 'Pantalones',
    'falda': 'Faldas', 'faldas': 'Faldas',
    'jersey': 'Jerseis', 'jerseis': 'Jerseis', 'jerséis': 'Jerseis',
    'abrigo': 'Abrigos', 'abrigos': 'Abrigos',
    'complementos': 'Accesorios', 'accesorio': 'Accesorios', 'accesorios': 'Accesorios'
}
productos['categoria'] = productos['categoria'].str.strip().str.lower().map(CANON)

# Columna temporada

TEMP = {
    'ss24': 'Verano 2024',
    'verano 2024': 'Verano 2024',
    'verano24': 'Verano 2024',
    'verano': 'Verano 2024',
}

productos['temporada'] = productos['temporada'].str.strip().str.lower().map(TEMP)

print(f'Categorias canonicas :{productos['categoria'].nunique()}')
print(f'Categoria con más productos: {productos['categoria'].value_counts().index[0]}')
print(f'Temporadas canónicas: {productos['temporada'].nunique()}')

Categorias canonicas :8
Categoria con más productos: Vestidos
Temporadas canónicas: 1


In [21]:
# Tabla visitas web
# Cambio de formato
visitas_web['fecha'] = pd.to_datetime(visitas_web['fecha'], format='mixed', dayfirst=True, errors='coerce')


print(f'Foramto de columna fecha: {visitas_web['fecha'].dtype}')
print(f"Rango de fechas: {visitas_web['fecha'].min().date()} a {visitas_web['fecha'].max().date()}")


Foramto de columna fecha: datetime64[us]
Rango de fechas: 2024-01-07 a 2024-12-07


# Exportar datos procesados

In [24]:
campaña.to_csv('../data/processed/campaña.csv')
clientes.to_csv('../data/processed/clientes.csv')
devoluciones.to_csv('../data/processed/devoluciones.csv')
lineas_pedido.to_csv('../data/processed/lineas_pedido.csv')
pedidos.to_csv('../data/processed/pedidos.csv')
productos.to_csv('../data/processed/productos.csv')
visitas_web.to_csv('../data/processed/visitas_web.csv')